# SPMV CSR

In [ ]:
from pynq import Overlay

#Load the overlay
overlay = Overlay('/home/xilinx/spmv_csr/spmv.bit')
print('Overlay loaded!')

In [ ]:
# Print all IPs found in the design
print(overlay.ip_dict.keys())

# Get handler of the spmv IP
spmv_ip = overlay.spmv_csr_0
print(spmv_ip)

# Print register map
print(spmv_ip.register_map)

In [ ]:
# Run the accelerator

import time
import numpy as np
from pynq import allocate

num_rows = 10
num_cols = 10

# CSR Data for the 10x10 sparse matrix
A_values_data    = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 1.0]
A_col_index_data = [0, 5, 1, 7, 2, 3, 8, 1, 4, 5, 9, 0, 6, 2, 7, 8, 9, 0, 9]
A_row_index_data = [0, 2, 4, 5, 7, 9, 11, 13, 15, 17, 19]

# Input vector 'x'
x_data           = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]

# Allocate physical memory in DDR
print('Allocating memory in DDR')

A_val_buf = allocate(shape=(len(A_values_data),), dtype=np.float32)
A_col_buf = allocate(shape=(len(A_col_index_data),), dtype=np.int32)
A_row_buf = allocate(shape=(len(A_row_index_data),), dtype=np.int32)

x_buf = allocate(shape=(num_cols,), dtype=np.float32)
y_buf = allocate(shape=(num_rows,), dtype=np.float32)

# Copy lists into buffers
A_val_buf[:] = A_values_data
A_col_buf[:] = A_col_index_data
A_row_buf[:] = A_row_index_data
x_buf[:]     = x_data

# Flush cache to ensure the FPGA reads the latest data from DDR
A_val_buf.sync_to_device()
A_col_buf.sync_to_device()
A_row_buf.sync_to_device()
x_buf.sync_to_device()

# Program hw registers

def write_64bit_address(ip, base_offset, address):
    """Writes a 64-bit physical memory address across two 32-bit registers."""
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)


ADDR_AP_CTRL    = 0x00
ADDR_NUM_ROWS   = 0x10
ADDR_NUM_COLS   = 0x18
ADDR_A_ROW_IDX  = 0x20
ADDR_A_COL_IDX  = 0x2c
ADDR_A_VALUES   = 0x38
ADDR_X          = 0x44
ADDR_Y          = 0x50

# Write scalars
spmv_ip.write(ADDR_NUM_ROWS, num_rows)
spmv_ip.write(ADDR_NUM_COLS, num_cols)

# Write memory pointers
write_64bit_address(spmv_ip, ADDR_A_ROW_IDX, A_row_buf.device_address)
write_64bit_address(spmv_ip, ADDR_A_COL_IDX, A_col_buf.device_address)
write_64bit_address(spmv_ip, ADDR_A_VALUES, A_val_buf.device_address)
write_64bit_address(spmv_ip, ADDR_X, x_buf.device_address)
write_64bit_address(spmv_ip, ADDR_Y, y_buf.device_address)

# Start SPMV engine

start_time = time.time()

# Write 1 to bit 0 of AP_CTRL to start the IP
spmv_ip.write(ADDR_AP_CTRL, 0x01)

# Poll bit 1 of AP_CTRL (ap_done) until it equals 1
while (spmv_ip.read(ADDR_AP_CTRL) & 0x02) == 0:
    pass

end_time = time.time()
print(f"HW Execution Time: {(end_time - start_time) * 1000:.4f} ms")

# Pull results back from DDR into CPU cache
y_buf.sync_from_device()

# Verify results

print("\n--- Verification ---")
y_sw = np.zeros(num_rows, dtype=np.float32)

# Calculate expected software result silently
for i in range(num_rows):
    row_start = A_row_index_data[i]
    row_end = A_row_index_data[i + 1]
    total = 0.0
    for j in range(row_start, row_end):
        total += A_values_data[j] * x_data[A_col_index_data[j]]
    y_sw[i] = total

# Print ONLY the resulting hardware vector
print("Resulting Hardware Vector (y):")
print(np.array(y_buf))

# Compare hardware vs software silently
match = True
for i in range(num_rows):
    if abs(y_buf[i] - y_sw[i]) >= 1e-5:
        match = False

if match:
    print("\n>>> SUCCESS! Hardware matches software. <<<")
else:
    print("\n>>> FAILED! Hardware does not match software. <<<")